# Evaluation of extremes of models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.default_params import *

In [ ]:
import functools
import itertools
import math
import string

import cf_xarray
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis import plot_map, STYLES, SUBREGIONS, BOX_LOCATIONS
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import plot_freq_density_figure, compute_metrics, DIST_THRESHOLDS, stat_bias, rms, plot_freq_density
from mlde_analysis.extremes import pred_and_target_return_times, plot_return_time_amounts
from mlde_utils import cp_model_rotated_pole, platecarree
from mlde_analysis import qq_plot, reasonable_quantiles

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, CPM_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

## Max bias

In [ ]:
for var in eval_vars:
    for normalize in [True, False]:
        max_biases = PRED_DAS[var].groupby("model").map(functools.partial(stat_bias, stat_func=xr.DataArray.max), cpm_da=VAR_DAS[var][f"target_{var}"], normalize=normalize)
        pretty_table(max_biases.groupby("model").map(rms), round=3)
        if normalize:
            style = STYLES[f"{var}Bias"]
        else:
            style = {}
        g = max_biases.plot(col="model", subplot_kws=dict(projection=cp_model_rotated_pole), **style)
        for ax in g.axs.flat:
            ax.coastlines()

## Q0.999 bias

In [ ]:
for var in eval_vars:
    for normalize in [True, False]:
        q999_biases = PRED_DAS[var].groupby("model").map(functools.partial(stat_bias, stat_func=functools.partial(xr.DataArray.quantile, q=0.999)), cpm_da=VAR_DAS[var][f"target_{var}"], normalize=normalize)
        pretty_table(q999_biases.groupby("model").map(rms), round=3)
        if normalize:
            style = STYLES[f"{var}Bias"]
        else:
            style = {}
        g = q999_biases.plot(col="model", subplot_kws=dict(projection=cp_model_rotated_pole), **style)
        for ax in g.axs.flat:
            ax.coastlines()

## Individual box locations

In [ ]:
fig = plt.figure(layout="constrained", figsize=(1.5, 1.5))
ax = fig.subplots(subplot_kw={"projection": cp_model_rotated_pole})
ax.coastlines(**{"resolution": "10m", "linewidth": 0.3})
da = CPM_DAS[eval_vars[0]]
ax.set_extent((
    da.cf["X"].min(),
    da.cf["X"].max(),
    da.cf["Y"].min(),
    da.cf["Y"].max(),
))

for label, q in BOX_LOCATIONS.items():
    single_box_da = da.cf.sel(**q, method="nearest")
    ax.plot(single_box_da.cf["X"].values, single_box_da.cf["Y"].values, color='blue', markersize=0.5, marker='o', transform=cp_model_rotated_pole)
    ax.annotate(
        xy=(single_box_da.cf["X"].item(), single_box_da.cf["Y"].item()), xycoords="data",
        text=label, xytext=(0, -15), textcoords="offset pixels",
        ha='center',va="center", transform=cp_model_rotated_pole, fontsize="xx-small")
    
plt.show()

## Figure: single box distribution

* Frequency Density Histogram of rainfall intensities

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
def _metrics(label, q, var_ds, thresholds):
    ds = var_ds.cf.sel(**q, method="nearest")
    cpm_da = ds[f"target_{var}"]
    pred_da = ds[f"pred_{var}"]
    
    return compute_metrics(pred_da, cpm_da, thresholds=thresholds).expand_dims({"location": [label]})
    
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
        
    metrics_ds = xr.concat(
        [ _metrics(label, q, VAR_DAS[var], DIST_THRESHOLDS[var]) for label, q in BOX_LOCATIONS.items() ], 
        dim="location"
    )

    pretty_table(metrics_ds, round=4)
    
    for label, q in BOX_LOCATIONS.items():
        ds = VAR_DAS[var].cf.sel(**q, method="nearest")
        pred_da = ds[f"pred_{var}"]
        cpm_da = ds[f"target_{var}"]
        
        fig = plt.figure(layout="constrained", figsize=(3.5, 2.5))
        
        ax = plot_freq_density_figure(pred_da, cpm_da, MODELLABEL2SPEC, fig)
        
        ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
        ax.set_title(label)
        
        plt.show()

### London vs SE (inc SE domain max) distribution

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    fig = plt.figure(layout="constrained", figsize=(7, 2.5))

    axd = fig.subplot_mosaic([["London", "SE", "SE max"]], sharex=True, sharey=True)

    
    label="London"
    q = BOX_LOCATIONS[label]
    
    ds = VAR_DAS[var].cf.sel(**q, method="nearest")
    
    pred_da = ds[f"pred_{var}"]
    cpm_da = ds[f"target_{var}"]
    
    hist_data = sorted(
        map(
            lambda modelgp: dict(
                data=modelgp[1].squeeze("model"),
                label=modelgp[0],
                color=MODELLABEL2SPEC[modelgp[0]]["color"],
            ),
            pred_da.groupby("model", squeeze=False),
        ),
        key=lambda x: MODELLABEL2SPEC[x["label"]]["order"],
    )

    ax = axd[label]
    plot_freq_density(hist_data, ax=ax, target_da=cpm_da, linewidth=1, yscale="log")
    
    # ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
    ax.set_title(label)

    label="SE"
    q = SUBREGIONS[label]
    
    ds = VAR_DAS[var]
    
    pred_da = ds[f"pred_{var}"]
    cpm_da = ds[f"target_{var}"]
    
    hist_data = sorted(
        map(
            lambda modelgp: dict(
                data=modelgp[1].squeeze("model"),
                label=modelgp[0],
                color=MODELLABEL2SPEC[modelgp[0]]["color"],
            ),
            pred_da.groupby("model", squeeze=False),
        ),
        key=lambda x: MODELLABEL2SPEC[x["label"]]["order"],
    )

    ax = axd[label]
    plot_freq_density(hist_data, ax=ax, target_da=cpm_da, linewidth=1, yscale="log", legend=False)
    
    # ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
    ax.set_title(label)
    
    label="SE"
    q = SUBREGIONS[label]
    
    ds = VAR_DAS[var].isel(**q).max(["grid_longitude", "grid_latitude"], keep_attrs=True)
    
    pred_da = ds[f"pred_{var}"]
    cpm_da = ds[f"target_{var}"]
    
    hist_data = sorted(
        map(
            lambda modelgp: dict(
                data=modelgp[1].squeeze("model"),
                label=modelgp[0],
                color=MODELLABEL2SPEC[modelgp[0]]["color"],
            ),
            pred_da.groupby("model", squeeze=False),
        ),
        key=lambda x: MODELLABEL2SPEC[x["label"]]["order"],
    )

    ax = axd[f"{label} max"]
    plot_freq_density(hist_data, ax=ax, target_da=cpm_da, linewidth=1, yscale="log", legend=False)
    
    # ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
    ax.set_title(f"{label} domain max")
        
    plt.show()

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    fig = plt.figure(layout="constrained", figsize=(7, 2.5))

    ax = fig.subplots()
    
    hist_data = [
        dict(
            data=VAR_DAS[var][f"target_{var}"].isel(**SUBREGIONS["SE"]),
            label="CPM SE",
            color="red",
        ),
        dict(
            data=VAR_DAS[var][f"target_{var}"].isel(**SUBREGIONS["SE"]).max(["grid_longitude", "grid_latitude"]),
            label="CPM SE max",
            color="orange",
        ),
        dict(
            data=VAR_DAS[var][f"target_{var}"].cf.sel(**BOX_LOCATIONS["London"], method="nearest"),
            label="CPM London",
            color="blue",
        ),
    ]

    plot_freq_density(hist_data, ax=ax, linewidth=1, yscale="log")
       
    plt.show()

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    fig = plt.figure(layout="constrained", figsize=(7, 1.5*len(ensemble_members)))

    axd = fig.subplot_mosaic(np.array(ensemble_members).reshape(-1,1), sharex=True, sharey=True)

    for i, em in enumerate(ensemble_members):
        da = VAR_DAS[var][f"target_{var}"].sel(ensemble_member=em)
        ax = axd[em]
        ax.plot(range(len(da["time"])), da.cf.sel(**BOX_LOCATIONS["London"], method="nearest"), linewidth=0.3, label="London")
        ax.plot(range(len(da["time"])), da.isel(**SUBREGIONS["SE"]).max(["grid_longitude", "grid_latitude"], keep_attrs=True), linewidth=0.3, label="SE max")
        if i == 0:
            ax.legend()


    plt.show()

### Lancaster vs NW (inc NW domain max) distribution

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    fig = plt.figure(layout="constrained", figsize=(7, 2.5))

    ax = fig.subplots()
    
    hist_data = [
        dict(
            data=VAR_DAS[var][f"target_{var}"].isel(**SUBREGIONS["NW"]),
            label="CPM NW",
            color="red",
        ),
        dict(
            data=VAR_DAS[var][f"target_{var}"].isel(**SUBREGIONS["NW"]).max(["grid_longitude", "grid_latitude"]),
            label="CPM NW max",
            color="orange",
        ),
        dict(
            data=VAR_DAS[var][f"target_{var}"].cf.sel(**BOX_LOCATIONS["Lancaster"], method="nearest"),
            label="CPM Lancaster",
            color="blue",
        ),
    ]

    plot_freq_density(hist_data, ax=ax, linewidth=1, yscale="log")
       
    plt.show()

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    fig = plt.figure(layout="constrained", figsize=(7, 1.5*len(ensemble_members)))

    axd = fig.subplot_mosaic(np.array(ensemble_members).reshape(-1,1), sharex=True, sharey=True)

    for i, em in enumerate(ensemble_members):
        da = VAR_DAS[var][f"target_{var}"].sel(ensemble_member=em)
        ax = axd[em]
        ax.plot(range(len(da["time"])), da.cf.sel(**BOX_LOCATIONS["Lancaster"], method="nearest"), linewidth=0.3, label="London")
        ax.plot(range(len(da["time"])), da.isel(**SUBREGIONS["NW"]).max(["grid_longitude", "grid_latitude"], keep_attrs=True), linewidth=0.3, label="SE max")
        if i == 0:
            ax.legend()


    plt.show()

## Figure: per time period single box distribution

* Frequency Density Histogram of rainfall intensities

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    
    metrics_ds = xr.concat([
            xr.concat([ _metrics(label, q, tp_ds, DIST_THRESHOLDS[var]) for label, q in BOX_LOCATIONS.items() ], dim="location").expand_dims({"time_period": [tp]})
         for tp, tp_ds in VAR_DAS[var].groupby("time_period") ], 
        dim="time_period"
    )

    pretty_table(metrics_ds, round=4)
    
    for label, q in BOX_LOCATIONS.items():
        ds = VAR_DAS[var].cf.sel(**q, method="nearest")
        for tp, tp_ds in ds.groupby("time_period"):
            pred_da = tp_ds[f"pred_{var}"]
            cpm_da = tp_ds[f"target_{var}"]
    
            fig = plt.figure(layout="constrained", figsize=(3.5, 2.5))
            ax = plot_freq_density_figure(pred_da, cpm_da, MODELLABEL2SPEC, fig)
            ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
            ax.set_title(f"{label} {tp}")
            
            plt.show()

## QQ plots

In [ ]:
quantile_dims=["ensemble_member", "time"]

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    for label, q in BOX_LOCATIONS.items():
        ds = VAR_DAS[var].cf.sel(**q, method="nearest")
        pred_da = ds[f"pred_{var}"]
        cpm_da = ds[f"target_{var}"]
        
        quantiles = reasonable_quantiles(cpm_da)
        cpm_quantiles = cpm_da.quantile(quantiles, dim=quantile_dims).rename("target_q")
    
        pred_quantiles = pred_da.quantile(quantiles, dim=quantile_dims).rename("pred_q")

        layout="constrained"

        fig, ax = plt.subplots(figsize=(5.5, 5.5), layout="constrained")

        xlabel = f"CPM \n{xr.plot.utils.label_from_attrs(da=cpm_da)}"
        ylabel = f"Predicted \n{xr.plot.utils.label_from_attrs(da=pred_da)}"

        qq_plot(ax, cpm_quantiles, pred_quantiles, title=f"Predicted quantiles vs CPM quantiles", xlabel=xlabel, ylabel=ylabel)

        plt.show()

## Return time plots (sorting): individual grid box

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_times(
                VAR_DAS[var].cf.sel(**q, method="nearest"), 
                var,
                n_days_per_year=360,
            ).expand_dims(location=[label])
            for label, q in BOX_LOCATIONS.items()
        ],
        dim="location",
    )

    plot_return_time_amounts(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()
        
    pretty_table(
        (rt_ds[f"pred_{var}_return_level"].sel(rp=[1, 10, 100], method="nearest") - rt_ds[f"target_{var}_return_level"].sel(rp=[1, 10, 100], method="nearest")).rename(f"{var} Return level errors"),
        round=3,
        pivot_table=dict(index=["location", "model", "rp"], columns="sample_id", values=f"{var} Return level errors")
    )

## Return time plots (sorting): individual grid box (seasonal)

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_times(
                VAR_DAS[var].cf.sel(**q, method="nearest").sel(time=(VAR_DAS[var]["time"]["time.season"] == season)),
                var,
                n_days_per_year=90
            ).expand_dims(location=[f"{label} {season}"])
            for (label, q), season in itertools.product(BOX_LOCATIONS.items(), ["DJF", "JJA"])
        ],
        dim="location",
    )

    plot_return_time_amounts(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()
        
    pretty_table(
        (rt_ds[f"pred_{var}_return_level"].sel(rp=[1, 10, 100], method="nearest") - rt_ds[f"target_{var}_return_level"].sel(rp=[1, 10, 100], method="nearest")).rename(f"{var} Return level errors"),
        round=3,
        pivot_table=dict(index=["location", "model", "rp"], columns="sample_id", values=f"{var} Return level errors")
    )

## Return time plots (sorting): subregions (domain max seasonal)

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_times(
                VAR_DAS[var].isel(**SUBREGIONS[srname]).sel(time=(CPM_DAS[var]["time"]["time.season"] == season)).max(dim=["grid_longitude", "grid_latitude"], keep_attrs=True),
                var,
                n_days_per_year=90,
            ).expand_dims(location=[f"{srname} {season}"])
            for (srname, season) in itertools.product(["NW", "SE"], ["DJF", "JJA"])
        ],
        dim="location",
    )
        
    plot_return_time_amounts(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()
        
    pretty_table(
        (rt_ds[f"pred_{var}_return_level"].sel(rp=[1, 10, 100], method="nearest") - rt_ds[f"target_{var}_return_level"].sel(rp=[1, 10, 100], method="nearest")).rename(f"{var} Return level errors"),
        round=3,
        pivot_table=dict(index=["location", "model", "rp"], columns="sample_id", values=f"{var} Return level errors")
    )